# NB34: newer predictor + molecular-pair error correction
Select **L4**, then **Run all**. Mount **bobofrogoo** Drive.

This is an experiment with published supporting evidence, **not a demonstrated 0.44 submission**. It tests TabPFN-3 alone, a simple neighbour correction, and a learned pair-error correction on the same five scaffold folds. The previous direct potency-difference experiment failed; this method corrects a strong baseline and reduces corrections when molecules are dissimilar.

[Method paper](https://arxiv.org/abs/2609.38744) reports gains on 46/58 molecular tasks, with modest average error reduction. This is an independent implementation; the linked author repository was unavailable. It uses our existing surrogate-ADME features rather than the paper's generic CheMeleon features. No challenge holdout or blind labels are packaged.

Each inner fold and inference batch is saved. Rerunning resumes completed batches. Changing code or inputs requires a new RUN name. No submission is uploaded, and no holdout evaluation occurs here. A small CPU end-to-end test and realistic global-context inference were checked locally; the full L4 accuracy/runtime result is still unmeasured.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess,sys,json,hashlib,zipfile,os,shutil,time
RUN='a'
BASE=Path('/content/drive/MyDrive')
OUT=BASE/'cyp_results'/f'nb34_{RUN}_pair_residual'
OUT.mkdir(parents=True,exist_ok=True)
print('Results and resume files:',OUT)


In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','-q','tabpfn==9.2.0','scikit-learn==1.7.2','numpy==2.1.3'])
import torch
assert torch.cuda.is_available(), 'Select an L4 GPU runtime first.'
print(torch.cuda.get_device_name(0), 'GPU GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,1))


In [ ]:
PAYLOAD=BASE/'cyp_payloads'/'nb34_pair_a0a9ea552077.zip'
assert PAYLOAD.exists(), f'Missing {PAYLOAD}. Mount bobofrogoo Drive.'
assert hashlib.file_digest(PAYLOAD.open('rb'),'sha256').hexdigest()=='deac4942429f4d40c816c485e45986099dd5324e148300cd5f5101c10a792c7e', 'Payload checksum mismatch'
ROOT=Path('/content/cyp_nb34_package');ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(PAYLOAD) as package:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in package.namelist())
    package.extractall(ROOT)
manifest=json.loads((ROOT/'package_manifest.json').read_text())
for name,expected in manifest.items():
    assert hashlib.file_digest((ROOT/name).open('rb'),'sha256').hexdigest()==expected,name
print(json.dumps(json.loads((ROOT/'input_audit.json').read_text()),indent=2))
shutil.copy2(ROOT/'plan.json',OUT/'plan.json')
shutil.copy2(ROOT/'input_audit.json',OUT/'input_audit.json')


In [ ]:
env=os.environ.copy()
env.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',TABPFN_DISABLE_TELEMETRY='1',TABPFN_ALLOW_CPU_LARGE_DATASET='1',HF_HUB_OFFLINE='1')
started=time.time()
# One model process at a time; first complete fold is also the real L4 resource pilot.
# If interrupted or out of memory, rerun this cell to reuse saved folds/batches.
command=[sys.executable,str(ROOT/'run_nb34.py'),'--root',str(ROOT),'--out',str(OUT),'--device','cuda']
with (OUT/'run.log').open('a',buffering=1) as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    try:
        for line in process.stdout:
            print(line,end='');log.write(line)
        rc=process.wait()
    except BaseException:
        process.terminate();process.wait();raise
    assert rc==0,f'Run stopped (exit {rc}); completed work is saved on Drive. See run.log.'
print('Run minutes:',round((time.time()-started)/60,1))


In [ ]:
subprocess.check_call([sys.executable,str(ROOT/'eval_nb34.py'),'--root',str(ROOT),'--out',str(OUT)],env=env)
report=json.loads((OUT/'development'/'development_results.json').read_text())
print('Development material gate:',report['material_gate_passed'])
for iso,result in report['endpoints'].items():
    print(iso,'best:',result.get('best'),'controls:',result.get('matched_standalone_controls'))
archive=BASE/'cyp_results'/f'nb34_{RUN}_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in OUT.rglob('*'):
        if path.is_file() and (path.suffix in ['.json','.log'] or path.name=='predictions.npz'):
            z.write(path,path.relative_to(OUT))
print('RESULTS SAVED:',archive)
print('Inner-fold and inference-batch resume caches remain separately on Drive.')
print('No holdout evaluated; no submission created or uploaded.')


If the development gate passes, retain the selected recipe and evaluate it once on the reserved holdout before making a submission. If it fails, keep the controls and diagnostics; a completed run is not automatically an improvement. The main check is improvement over the existing ensemble, plus the pair method's performance against its own matched global control.